In [1]:
import pandas as pd
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DoomGuy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = pd.read_csv('../data/raw/SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
df.drop_duplicates(subset="message", inplace=True)

def clean(message):
    message = message.lower()
    message = message.translate(str.maketrans("", "", string.punctuation))
    words = [stemmer.stem(word) for word in message.split() if word not in stop_words]
    return " ".join(words)
df["cleaned"] = df["message"].apply(clean)


In [3]:
X = df["cleaned"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [4]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

y_pred[:20]

array(['ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham',
       'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham',
       'ham', 'ham'], dtype='<U4')

In [5]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       903
        spam       1.00      0.73      0.84       131

    accuracy                           0.97      1034
   macro avg       0.98      0.86      0.91      1034
weighted avg       0.97      0.97      0.96      1034

[[903   0]
 [ 36  95]]


Naive Bayes baseline: spam F1 = 0.84 (precision 1.00, recall 0.73). Perfect precision (no false positives) but misses 36/131 spam which is characteristic for NB profile being conservative and high-precision. Baseline for later models to be evaluated on.

In [6]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr_balanced = LogisticRegression(class_weight='balanced', max_iter=1000)

lr.fit(X_train_tfidf, y_train)
lr_balanced.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)
y_pred_lr_balanced = lr_balanced.predict(X_test_tfidf)

print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

print(classification_report(y_test, y_pred_lr_balanced))
print(confusion_matrix(y_test, y_pred_lr_balanced))

              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       903
        spam       0.97      0.70      0.81       131

    accuracy                           0.96      1034
   macro avg       0.96      0.85      0.90      1034
weighted avg       0.96      0.96      0.96      1034

[[900   3]
 [ 39  92]]
              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       903
        spam       0.87      0.90      0.88       131

    accuracy                           0.97      1034
   macro avg       0.93      0.94      0.93      1034
weighted avg       0.97      0.97      0.97      1034

[[885  18]
 [ 13 118]]


Logistic Regression has F1 of 0.81, underperforming Naive Bayes baseline. With class_weight='balanced', F1 rises to 0.88 with spam recall increasing to 0.90 but precision drops to 0.87. Best F1 so far but for deployment, choice depends on tradeoff between false positives and false negatives.

In [7]:
from sklearn.svm import LinearSVC

svc = LinearSVC()
svc_balanced = LinearSVC(class_weight='balanced')

svc.fit(X_train_tfidf, y_train)
svc_balanced.fit(X_train_tfidf,y_train)

y_pred_svm = svc.predict(X_test_tfidf)
y_pred_svm_balanced = svc_balanced.predict(X_test_tfidf)

print(classification_report(y_test, y_pred_svm))
print(confusion_matrix(y_test, y_pred_svm))

print(classification_report(y_test, y_pred_svm_balanced))
print(confusion_matrix(y_test, y_pred_svm_balanced))

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       903
        spam       0.95      0.89      0.92       131

    accuracy                           0.98      1034
   macro avg       0.97      0.94      0.95      1034
weighted avg       0.98      0.98      0.98      1034

[[897   6]
 [ 15 116]]
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       903
        spam       0.90      0.91      0.90       131

    accuracy                           0.98      1034
   macro avg       0.94      0.95      0.95      1034
weighted avg       0.98      0.98      0.98      1034

[[890  13]
 [ 12 119]]


Linear SVC has spam F1 of 0.92, performing better than Naive Bayes baseline and the best classical model. Recall 0.89 (misses 15/131) with precision 0.95 (6 false positives). Balancing recall and precision better than LogReg-balanced which achieved somewhat similar recall at the cost of 3x more false-positives. Confirms linear SVMs' known strength on sparse text features.

In [10]:
results = []
for name, preds in [("Naive Bayes", y_pred), ("LogReg", y_pred_lr), ("Balanced LogReg", y_pred_lr_balanced), ("SVM", y_pred_svm), ("Balanced SVM", y_pred_svm_balanced)]:
    rep = classification_report(y_test, preds, output_dict=True)
    spam = rep['spam']
    results.append({"model": name, "precision": spam['precision'], "recall": spam['recall'], "F1": spam['f1-score']})
comparison = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True).round(3)
comparison

,model,precision,recall,F1
0,SVM,0.951,0.885,0.917
1,Balanced SVM,0.902,0.908,0.905
2,Balanced LogReg,0.868,0.901,0.884
3,Naive Bayes,1.000,0.725,0.841
4,LogReg,0.968,0.702,0.814


The best model is SVM with a spam F1 of 0.917 while also being very strong overall with a precision of 0.95 and a recall of 0.89. This contrasts with Naive Bayes which has a precision of 1.00 but a much lower recall of 0.725 and a F1 of 0.841. The tradeoff, and the decision of what to prioritize, is important to think about because simply choosing the largest F1 isn't necessarily the best choice. While the Naive Bayes model did miss classifying 27% of spam, it never wrongly classified ham as spam which for some is preferable as having actual messages being wrongly blocked can be fairly costly. However, I would choose the SVM as it has the strongest overall performance of a model. That being said, if lowering false positives were a priority, I would choose Naive Bayes.